# Sensitivity of SPB Targets to Data and Assumptions

This notebook asks how much the SPB targets of the EU fiscal rules depend on the data and the model assumptions behind them, and whether the baseline assumptions sit at the restrictive or the accommodative end of the range of plausible alternatives. The baseline is close to the reference trajectories of the Commission prior guidance of 2024: it uses the input workbook built from the Commission calculation sheets and the Commission implementation of the rules (`rules='commission'`), which is kept in all runs. All runs use the fiscal multiplier with a persistent effect on the output gap (`fiscal_multiplier_type='pers'`), whose persistence is one of the checks, instead of the output gap closure rule of the Commission sheets. The text, charts and tables refer to the 4-year adjustment period; the annex summarises the individual checks for the 7-year adjustment period.

Each run records two targets:

- the **DSA target**: the SPB required by the DSA criteria alone (debt declines or stays below 60% in the deterministic scenarios, deficit below 3%, stochastic criterion);
- the **binding target**: the SPB after the excessive deficit procedure (EDP), the debt sustainability safeguard and the deficit resilience safeguard are added.

Most assumptions work through the DSA criteria. Where the EDP or a safeguard binds, it can hide the effect of an assumption or add effects of its own; the tables and charts show both targets.

1. **Individual checks.** Each check changes one data input or model assumption, one at a time. The checks cover data revisions, the repayment profile of debt, macroeconomic assumptions, the fiscal multiplier, the calibration of the stress tests and the stochastic analysis. Runs with nine other random seeds give a noise band.
2. **Global sensitivity analysis.** Random draws of all model assumptions of the individual checks at once (data revisions excluded). For each country, the lowest and the highest draw give the most accommodative and the most restrictive combination of assumptions; a variance decomposition shows which assumptions matter.
3. **Main findings**, including where the baseline sits within the range of alternatives.
4. **Results tables**, displayed and saved to `output/sensitivity/`.
5. **Annex.** Individual checks for a 7-year adjustment period.

Each run records the target, the annual adjustment, the binding criterion, the SPB target of each criterion, net expenditure growth, debt 10 years after the adjustment period, and the gap to the next criterion: how far the target would fall if the binding criterion were dropped. Both targets come from the same run: `find_spb_binding` finds the DSA target first and then applies the EDP and the safeguards.

Runs are cached in `output/sensitivity/`: with `RUN = True`, the notebook computes only runs that are not in the cache. The individual checks take about 50 minutes on 12 cores, the global draws about 1.5 hours (all runs include the stochastic criterion, about 10 seconds per run). `QUICK = True` runs a subset of countries and fewer global draws for testing. The helper functions are in `code/functions/sensitivity.py`.

For comments and suggestions please contact lennard.welslau[at]gmail[dot]com.\
Last update: October 2026

In [ ]:
# Import libraries and modules
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
plt.rcParams.update({'axes.grid': True, 'grid.color': 'black', 'grid.alpha': 0.15, 'grid.linestyle': '--', 'font.size': 12,
                     'figure.dpi': 100, 'axes.titleweight': 'bold'})

# Import the DSA model classes and functions from the code folder
sys.path.append(os.path.abspath('..'))
from classes import StochasticDsaModel as DSA
from data_pipeline import EU27, latest_input_file
from functions.sensitivity import *

# Set autoreload
%load_ext autoreload
%autoreload 2

## Settings

In [ ]:
INPUT_FILE = 'dsa_inputs_commission_2024.xlsx'  # baseline input workbook (Commission prior guidance 2024)
BASELINE_RULES = 'commission'                    # rules of find_spb_binding in the baseline
MULTIPLIER_TYPE = 'pers'                         # fiscal multiplier with persistent effect in all runs (None: input file)
ADJUSTMENT_PERIODS = (4, 7)                      # adjustment periods of the individual checks
MAIN_PERIOD = 4                                  # period of the global analysis, text, charts and tables (others: annex)
COUNTRIES = EU27
SEED = 0                                         # random seed set before every run (noise band: seeds 1 to 9)
N_DRAWS = 200                                    # draws of the global sensitivity analysis
MAX_WORKERS = None                               # worker processes, None uses all CPUs

QUICK = False                                    # True: subset of countries and fewer global draws (for testing)
QUICK_COUNTRIES = ['BEL', 'DEU', 'ESP', 'FRA', 'ITA', 'NLD']
RUN = True                                       # True: compute runs missing from the cache; False: use cached runs only

if QUICK:
    COUNTRIES, N_DRAWS = QUICK_COUNTRIES, 40

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_options = dict(countries=COUNTRIES, adjustment_periods=ADJUSTMENT_PERIODS, input_file=INPUT_FILE,
                   rules=BASELINE_RULES, seed=SEED, run=RUN, max_workers=MAX_WORKERS,
                   model_kwargs={'fiscal_multiplier_type': MULTIPLIER_TYPE} if MULTIPLIER_TYPE else None)

## 1. Individual checks

### 1.1 Specification table

Each row changes one setting relative to the baseline. Starting values such as the SPB, the balances and debt in T are data, not assumptions, and are not varied; the data revision check instead reruns all countries on the latest input workbook, in which the forecasts for T and the following years of the 2024 guidance are replaced by outturns. The repayment profile check replaces the share of long-term debt maturing each year by a repayment profile of the long-term debt outstanding in T (`bond_data=True`), built from Eurostat data on debt by residual maturity at the end of T-1 (`gov_10dd_rmd`, see `data_pipeline/maturity.py`) and scaled to long-term debt in T. Hungary, Luxembourg and the Netherlands do not report the data, so these runs fail for them (shown as missing). Input changes are relative to the baseline workbook: the notebook reads the baseline value with `read_country` and passes the new absolute value as an override. Potential growth is shifted together with baseline real growth, so that the output gap is unchanged. The ageing-cost check scales the change in ageing costs relative to T; the stock-flow check sets the stock-flow adjustment to zero after T.

Interest rate and inflation checks shift the model paths as if the T+10 or T+30 anchor were different, with the linear interpolation the model uses between anchors (`shift_anchor`). Changing the anchor parameters in the workbook would not work here: the Commission workbook provides full paths of market rates and inflation up to about T+20, and the model uses the anchor parameters only to fill years without data.

In [ ]:
repayment_profiles = eurostat_repayment_profiles(INPUT_FILE, COUNTRIES)  # Eurostat data (fetched once)
profiles_path = save_repayment_profiles(repayment_profiles)               # file read by the global draws
oat_specs = build_oat_specs(latest_file=latest_input_file(), repayment_profiles=repayment_profiles)
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 70):
    display(spec_table(oat_specs))

### 1.2 Runs

The baseline and every check run for each country and adjustment period. The Commission targets are added for comparison: the published SPB at the end of the adjustment period (after the safeguards) and the target from the DSA criteria alone.

In [ ]:
oat = run_specs(oat_specs, cache_file=OUTPUT_DIR / 'oat_runs.pkl', **run_options)
vintage = 'latest' if 'latest' in INPUT_FILE else '2024'
oat = oat.merge(commission_targets(COUNTRIES, vintage=vintage), on=['country', 'adjustment_period'], how='left')
oat = add_deviations(oat)
noise = noise_band(oat, target='dsa')
checks = oat[oat['group'] != 'Baseline']
print(f"{len(oat)} runs, {oat['error'].notna().sum() if 'error' in oat else 0} failed")

### 1.3 Baseline and Commission targets

The baseline is compared with the Commission targets, both from the DSA criteria alone and after the safeguards. The model reproduces them closely with the Commission's output gap closure rule (`MULTIPLIER_TYPE = None`, see `commission_replication.ipynb`); the persistent multiplier used here moves some targets by a few tenths of a percentage point.

In [ ]:
base = oat[(oat['id'] == 'baseline') & (oat['adjustment_period'] == MAIN_PERIOD)].copy()
base['dsa difference'] = base['dsa_target'] - base['commission_dsa_target']
base['binding difference'] = base['binding_target'] - base['commission_target']
table = base.set_index('country')[['commission_dsa_target', 'dsa_target', 'dsa difference', 'commission_target',
                                   'binding_target', 'binding difference', 'binding_criterion']]
display(table.round(2))
print(f"Median absolute difference {base['dsa difference'].abs().median():.2f} pp. (DSA target), "
      f"{base['binding difference'].abs().median():.2f} pp. (binding target); "
      f"maximum {base['dsa difference'].abs().max():.2f} and {base['binding difference'].abs().max():.2f} pp.")

### 1.4 Main table: effects of each check

Change of the SPB target relative to the baseline for each check variant, in pp. of GDP: mean, minimum and maximum across countries and the number of countries whose target changes by more than 0.1 pp. (whether an effect is broad or driven by a few countries), for the DSA target (DSA criteria only) and the binding target (DSA criteria, EDP and safeguards). The noise band from alternative seeds is printed above the table; effects within it are indistinguishable from simulation noise. Countries without data for a check (no Eurostat repayment profile for Hungary, Luxembourg and the Netherlands) are left out of its statistics.

In [ ]:
main_table = effects_table(oat, (MAIN_PERIOD,))
main_noise = noise[noise['adjustment_period'] == MAIN_PERIOD]
print(f"Noise band (median half-width across countries, DSA target): {float((main_noise['noise_high'] - main_noise['noise_low']).median() / 2):.3f} pp.")
with pd.option_context('display.max_rows', 200):
    display(main_table.round(2))

### 1.5 Which checks matter most

Mean change of the SPB target across countries for each check variant: bars for the DSA target, diamonds for the binding target. Means keep the sign of the changes, so a check that raises the target in some countries and lowers it in others can have a small mean; the range across countries is in the main table.

In [ ]:
plot_mean_effects(oat, (MAIN_PERIOD,))
plt.show()

## 2. Global sensitivity analysis

Individual checks miss interactions: a higher interest rate may matter more once growth is low. The global analysis draws all model assumptions of the individual checks at once (Latin hypercube): continuous assumptions uniformly over the range of their values in the individual checks, discrete settings (ageing cost period, stock-flow adjustment, repayment profile, multiplier persistence and the settings of the stochastic analysis) with equal probability for each setting, the baseline setting included. Data revisions and the random seed are not drawn. Each draw is a full run with the stochastic criterion, for the 4-year adjustment period.

For each country, the draw with the lowest target is the most accommodative and the draw with the highest target the most restrictive combination of assumptions, separately for the DSA target and the binding target. The table shows their change relative to the baseline, as in the main table (section 1.4).

In [ ]:
global_specs = build_global_specs(n_draws=N_DRAWS, seed=SEED, profiles_path=profiles_path)
glob = run_specs(global_specs, cache_file=OUTPUT_DIR / 'global_runs.pkl', **{**run_options, 'adjustment_periods': (MAIN_PERIOD,)})
glob = add_deviations(glob)
combinations = draw_combinations(glob)
results = add_deviations(pd.concat([oat, combinations], ignore_index=True))
combined_table = effects_table(combinations, (MAIN_PERIOD,))
display(combined_table.round(2))

### 2.1 Which assumptions matter

The variance of the target across draws is split by country with squared standardised regression coefficients: the share of the variance explained linearly by each assumption (discrete settings with more than two values enter as dummies). The draws are close to uncorrelated, so the shares add up to the R² of the regression; the rest is due to non-linear effects and interactions, mostly switches of the binding criterion. The table aggregates over countries, weighting each country by the variance of its target, so the shares refer to the variance summed over countries; the last row is the mean standard deviation of the target across countries.

In [ ]:
draws = glob[glob['group'] == 'Global']
vd = pd.concat([variance_decomposition(draws, target='dsa'), variance_decomposition(draws, target='binding')],
               ignore_index=True)
variance = variance_table(vd)
display(variance.round(2))

### 2.2 Range of targets by country

Targets by country, ordered by the baseline target. The grey bar spans the targets of the individual checks of model assumptions (data revisions and seeds excluded); green and red markers show the lowest and the highest draw of the global analysis. Markers also show the baseline, the Commission target and the SPB in T; the distance between the SPB in T and a target is the adjustment the target requires. DSA targets first, then binding targets (Commission target as published).

In [ ]:
for target in ['dsa', 'binding']:
    plot_ranges(results, MAIN_PERIOD, target=target)
    plt.show()

### 2.3 Debt paths by country

Debt under a linear adjustment to the DSA target of the baseline and of the lowest and highest draw (green, red), with the range across the individual checks of model assumptions in grey.

In [ ]:
plot_debt_paths(results, adjustment_period=MAIN_PERIOD, target='dsa')
plt.show()

## 3. Main findings

Written from the full run of the global analysis: where the baseline (Commission assumptions and rules) sits within the range of the individual checks and the global draws, which assumptions matter most, and how the EDP and the safeguards change the picture.

## 4. Results tables

The main result tables, also saved to `output/sensitivity/sensitivity_results.xlsx` together with the tidy results (one row per run, sheets `runs` and `global_runs`). Columns with dictionaries (SPB targets by criterion, debt paths) are kept in the cached runs (`*.pkl`).

In [ ]:
# Result tables
tables = {
    'Baseline and Commission targets (1.3)': table,
    'Effects of each individual check (1.4)': main_table,
    'Lowest and highest draw of the global analysis (2)': combined_table,
    'Variance decomposition of the global analysis (2.1)': variance,
    'Specifications of the individual checks': spec_table(oat_specs),
}
for title, t in tables.items():
    display(Markdown(f'**{title}**'))
    with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 70):
        display(t.round(2))

# Save, with the annex table (section 5)
annex = effects_table(oat, tuple(n for n in ADJUSTMENT_PERIODS if n != MAIN_PERIOD))
flat = lambda df: df.drop(columns=[c for c in ['spb_target_dict', 'debt_path', 'dsa_debt_path'] if c in df])
sheets = {'effects': main_table, 'annex_effects': annex, 'global_combinations': combined_table,
          'global_variance': variance, 'baseline_commission': table, 'specs': tables['Specifications of the individual checks']}
with pd.ExcelWriter(OUTPUT_DIR / 'sensitivity_results.xlsx') as writer:
    flat(results).to_excel(writer, sheet_name='runs', index=False)
    for name, t in sheets.items():
        t.to_excel(writer, sheet_name=name, index=not isinstance(t.index, pd.RangeIndex))
    flat(glob).to_excel(writer, sheet_name='global_runs', index=False)
    vd.to_excel(writer, sheet_name='global_variance_by_country', index=False)
print(f"Saved to {OUTPUT_DIR / 'sensitivity_results.xlsx'}")

## 5. Annex: 7-year adjustment

Effects of each individual check on the SPB target for a 7-year adjustment period, as in the main table (section 1.4): mean, minimum and maximum change relative to the baseline across countries and the number of countries with a change above 0.1 pp., for the DSA target and the binding target (pp. of GDP). The global analysis covers the 4-year adjustment period only.

In [ ]:
with pd.option_context('display.max_rows', 200):
    display(annex.round(2))